# 03_inference.ipynb

## One-sample inference for Project Cycle 2

This notebook performs inference for:

1. **Behavior variable:** `CurrentCigaretteUse`
   - one-sample proportion confidence interval
   - one-sample proportion hypothesis test

2. **Continuous variable:** `BMIPCT`
   - one-sample mean confidence interval
   - one-sample one-sample t-test

## Research questions

### Proportion analysis
Is the proportion of students who currently use cigarettes different from **0.20**?

### Mean analysis
Is the mean BMI percentile (`BMIPCT`) different from **65.0**?

## Why one-sample inference is appropriate

We use **one-sample inference** when we compare a sample result to a single benchmark value.

- For `CurrentCigaretteUse`, we compare the sample proportion to **p0 = 0.20**
- For `BMIPCT`, we compare the sample mean to **μ0 = 65.0**

We are not comparing two separate groups, so one-sample methods are the correct tools.

In [1]:
import os
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats

In [2]:
# Create required output folders
os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/summary", exist_ok=True)

In [3]:
# Load processed and recoded data
df = pd.read_csv("../data/processed/yrbs_recoded.csv")

# Keep only needed variables
required_cols = ["CurrentCigaretteUse", "BMIPCT"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

display(df[required_cols].head())
print("Dataset shape:", df.shape)

,CurrentCigaretteUse,BMIPCT
0,1.0,NaN
1,NaN,66.531824
2,NaN,NaN
3,0.0,98.174319
4,0.0,NaN


Dataset shape: (14041, 103)


## Variable setup and benchmark values

### Behavior variable
- Variable: `CurrentCigaretteUse`
- Recoded meaning:
  - **1 = success** = current cigarette use
  - **0 = failure** = no current cigarette use
- Benchmark proportion: **p0 = 0.20**

### Continuous variable
- Variable: `BMIPCT`
- Benchmark mean: **μ0 = 65.0**

In [4]:
# Analysis datasets
behavior = df["CurrentCigaretteUse"].dropna()
continuous = df["BMIPCT"].dropna()

# Benchmarks
p0 = 0.20
mu0 = 65.0

print("Behavior n:", len(behavior))
print("Continuous n:", len(continuous))

Behavior n: 13323
Continuous n: 13062


## Part 1: One-sample inference for the population proportion

### Parameter of interest
Let **p** be the population proportion of students who currently use cigarettes.

### Research question
Is the population proportion of current cigarette use different from **0.20**?

### Hypotheses
- **Null hypothesis:** H0: p = 0.20
- **Alternative hypothesis:** HA: p ≠ 0.20

### Why a one-sample proportion test?
We use a one-sample proportion test because:
- the variable is binary after recoding,
- the sample gives a proportion of successes,
- and we are comparing that one sample proportion to a single benchmark value, **0.20**.

In [5]:
# Basic sample results for the proportion analysis
n_prop = len(behavior)
x_success = int(behavior.sum())
p_hat = behavior.mean()

prop_basic = pd.DataFrame({
    "quantity": ["n", "successes", "sample proportion", "benchmark proportion"],
    "value": [n_prop, x_success, p_hat, p0]
})

prop_basic

,quantity,value
0,n,13323.000000
1,successes,2589.000000
2,sample proportion,0.194326
3,benchmark proportion,0.200000


In [6]:
# Check large-count condition for normal-based CI/test
expected_successes = n_prop * p0
expected_failures = n_prop * (1 - p0)

condition_table = pd.DataFrame({
    "condition": ["n * p0", "n * (1 - p0)"],
    "value": [expected_successes, expected_failures],
    "meets_rule_of_thumb_(>=10)": [expected_successes >= 10, expected_failures >= 10]
})

condition_table

,condition,value,meets_rule_of_thumb_(>=10)
0,n * p0,2664.6,True
1,n * (1 - p0),10658.4,True


### Why we check conditions

For the one-sample proportion procedures, we want the sample size to be large enough for the sampling distribution of the sample proportion to be approximated reasonably well.

A common rule is to check:
- n × p0 ≥ 10
- n × (1 − p0) ≥ 10

If these are satisfied, the z-based one-sample proportion test is generally reasonable.

In [7]:
# 95% confidence interval for population proportion using normal approximation
alpha = 0.05
z_star = stats.norm.ppf(1 - alpha/2)

se_phat = math.sqrt((p_hat * (1 - p_hat)) / n_prop)
ci_prop_lower = p_hat - z_star * se_phat
ci_prop_upper = p_hat + z_star * se_phat

# Bound CI to [0,1]
ci_prop_lower = max(0, ci_prop_lower)
ci_prop_upper = min(1, ci_prop_upper)

print(f"Sample proportion (p-hat): {p_hat:.4f}")
print(f"95% CI for p: ({ci_prop_lower:.4f}, {ci_prop_upper:.4f})")

Sample proportion (p-hat): 0.1943
95% CI for p: (0.1876, 0.2010)


In [8]:
# One-sample z test for population proportion
se_null = math.sqrt((p0 * (1 - p0)) / n_prop)
z_stat = (p_hat - p0) / se_null
p_value_prop = 2 * (1 - stats.norm.cdf(abs(z_stat)))

prop_test_results = pd.DataFrame({
    "statistic": ["z", "p-value"],
    "value": [z_stat, p_value_prop]
})

prop_test_results

,statistic,value
0,z,-1.637423
1,p-value,0.101542


In [9]:
# Decision for proportion test
alpha = 0.05
prop_decision = "Reject H0" if p_value_prop < alpha else "Fail to reject H0"

print("One-sample proportion test")
print(f"H0: p = {p0}")
print(f"HA: p != {p0}")
print(f"z = {z_stat:.4f}")
print(f"p-value = {p_value_prop:.6f}")
print(f"Decision at alpha = 0.05: {prop_decision}")

One-sample proportion test
H0: p = 0.2
HA: p != 0.2
z = -1.6374
p-value = 0.101542
Decision at alpha = 0.05: Fail to reject H0


### Interpretation of the one-sample proportion test
There is not sufficient statistical evidence to conclude that the population proportion of students who currently use cigarettes is different from 0.20.

## Part 2: One-sample inference for the population mean

### Parameter of interest
Let **μ** be the population mean of `BMIPCT`.

### Research question
Is the population mean of `BMIPCT` different from **65.0**?

### Hypotheses
- **Null hypothesis:** H0: μ = 65.0
- **Alternative hypothesis:** HA: μ ≠ 65.0

In [10]:
# Basic sample results for the mean analysis
n_mean = len(continuous)
xbar = continuous.mean()
s = continuous.std(ddof=1)

mean_basic = pd.DataFrame({
    "quantity": ["n", "sample mean", "sample standard deviation", "benchmark mean"],
    "value": [n_mean, xbar, s, mu0]
})

mean_basic

,quantity,value
0,n,13062.000000
1,sample mean,64.820683
2,sample standard deviation,27.516756
3,benchmark mean,65.000000


In [11]:
# 95% confidence interval for the population mean
dfree = n_mean - 1
t_star = stats.t.ppf(1 - alpha/2, df=dfree)
se_mean = s / math.sqrt(n_mean)

ci_mean_lower = xbar - t_star * se_mean
ci_mean_upper = xbar + t_star * se_mean

print(f"Sample mean: {xbar:.4f}")
print(f"Sample SD: {s:.4f}")
print(f"95% CI for mean BMIPCT: ({ci_mean_lower:.4f}, {ci_mean_upper:.4f})")

Sample mean: 64.8207
Sample SD: 27.5168
95% CI for mean BMIPCT: (64.3487, 65.2926)


In [12]:
# One-sample t-test for the population mean
t_stat, p_value_mean = stats.ttest_1samp(continuous, popmean=mu0, nan_policy="omit")

mean_test_results = pd.DataFrame({
    "statistic": ["t", "p-value"],
    "value": [t_stat, p_value_mean]
})

mean_test_results

,statistic,value
0,t,-0.744781
1,p-value,0.456417


In [13]:
# Decision for mean test
mean_decision = "Reject H0" if p_value_mean < alpha else "Fail to reject H0"

print("One-sample t-test")
print(f"H0: mu = {mu0}")
print(f"HA: mu != {mu0}")
print(f"t = {t_stat:.4f}")
print(f"p-value = {p_value_mean:.6f}")
print(f"Decision at alpha = 0.05: {mean_decision}")

One-sample t-test
H0: mu = 65.0
HA: mu != 65.0
t = -0.7448
p-value = 0.456417
Decision at alpha = 0.05: Fail to reject H0


### Interpretation of the one-sample t-test
Based on this sample, the average BMI percentile of students appears to be consistent with 65.0, and there is no strong statistical evidence of a meaningful deviation from this benchmark.

In [14]:
# Build inference summary table
inference_summary = pd.DataFrame({
    "analysis": [
        "Proportion", "Proportion", "Proportion", "Proportion", "Proportion", "Proportion", "Proportion",
        "Mean", "Mean", "Mean", "Mean", "Mean", "Mean", "Mean"
    ],
    "metric": [
        "Variable",
        "n",
        "Sample estimate",
        "Benchmark",
        "95% CI lower",
        "95% CI upper",
        "Hypothesis test decision",
        "Variable",
        "n",
        "Sample estimate",
        "Benchmark",
        "95% CI lower",
        "95% CI upper",
        "Hypothesis test decision"
    ],
    "value": [
        "CurrentCigaretteUse",
        int(n_prop),
        float(round(p_hat, 6)),
        float(p0),
        float(round(ci_prop_lower, 6)),
        float(round(ci_prop_upper, 6)),
        prop_decision,
        "BMIPCT",
        int(n_mean),
        float(round(xbar, 6)),
        float(mu0),
        float(round(ci_mean_lower, 6)),
        float(round(ci_mean_upper, 6)),
        mean_decision
    ]
})

inference_summary

,analysis,metric,value
0,Proportion,Variable,CurrentCigaretteUse
1,Proportion,n,13323
2,Proportion,Sample estimate,0.194326
3,Proportion,Benchmark,0.2
4,Proportion,95% CI lower,0.187607
5,Proportion,95% CI upper,0.201044
6,Proportion,Hypothesis test decision,Fail to reject H0
7,Mean,Variable,BMIPCT
8,Mean,n,13062
9,Mean,Sample estimate,64.820683


In [15]:
# Add more detailed inferential statistics table
inference_detailed = pd.DataFrame({
    "analysis": ["Proportion", "Proportion", "Mean", "Mean"],
    "metric": ["test_statistic", "p_value", "test_statistic", "p_value"],
    "value": [
        float(round(z_stat, 6)),
        float(round(p_value_prop, 6)),
        float(round(t_stat, 6)),
        float(round(p_value_mean, 6))
    ]
})

inference_summary_full = pd.concat([inference_summary, inference_detailed], ignore_index=True)
inference_summary_full

,analysis,metric,value
0,Proportion,Variable,CurrentCigaretteUse
1,Proportion,n,13323
2,Proportion,Sample estimate,0.194326
3,Proportion,Benchmark,0.2
4,Proportion,95% CI lower,0.187607
5,Proportion,95% CI upper,0.201044
6,Proportion,Hypothesis test decision,Fail to reject H0
7,Mean,Variable,BMIPCT
8,Mean,n,13062
9,Mean,Sample estimate,64.820683


In [16]:
# Save inference summary table
inference_summary_full.to_csv("../outputs/tables/inference_summary_table.csv", index=False)

print("Saved: ../outputs/tables/inference_summary_table.csv")

Saved: ../outputs/tables/inference_summary_table.csv


In [22]:
# Build final written summary
prop_ci_contains_p0 = (ci_prop_lower <= p0 <= ci_prop_upper)
mean_ci_contains_mu0 = (ci_mean_lower <= mu0 <= ci_mean_upper)

final_summary_text = f"""
Project Cycle 2 Final Summary
=============================

Research Questions
------------------
1. Is the proportion of students who currently use cigarettes different from 0.20?
2. Is the mean BMIPCT different from 65.0?

Behavior Variable: CurrentCigaretteUse
--------------------------------------
Parameter: population proportion p
Benchmark: p0 = {p0:.2f}

Sample size: {n_prop}
Sample proportion: {p_hat:.4f}
95% confidence interval: ({ci_prop_lower:.4f}, {ci_prop_upper:.4f})
Test statistic (z): {z_stat:.4f}
P-value: {p_value_prop:.6f}
Decision at alpha = 0.05: {prop_decision}

Interpretation:
The sample proportion of students who currently use cigarettes was estimated as {p_hat:.4f}. The 95% confidence interval ({ci_prop_lower:.4f}, {ci_prop_upper:.4f}) provides a range of plausible values for the true population proportion.

{"Because the benchmark value 0.20 lies within this confidence interval and the p-value is greater than 0.05, we do not have sufficient evidence to conclude that the population proportion differs from 0.20." if prop_ci_contains_p0 else "Because the benchmark value 0.20 does not lie within this confidence interval and the p-value is less than 0.05, we have sufficient evidence to conclude that the population proportion differs from 0.20."}

This result is consistent with the EDA findings, where the observed sample proportion was already very close to 0.20. Although the additional age-based EDA suggested that current cigarette use tends to increase with age, the overall proportion in the full sample remains close to the benchmark value.

Therefore, the proportion of students who currently use cigarettes appears to be consistent with 0.20, and the small observed difference is likely due to sampling variability.

Continuous Variable: BMIPCT
---------------------------
Parameter: population mean mu
Benchmark: mu0 = {mu0:.1f}

Sample size: {n_mean}
Sample mean: {xbar:.4f}
Sample standard deviation: {s:.4f}
95% confidence interval: ({ci_mean_lower:.4f}, {ci_mean_upper:.4f})
Test statistic (t): {t_stat:.4f}
P-value: {p_value_mean:.6f}
Decision at alpha = 0.05: {mean_decision}

Interpretation:
The sample mean BMIPCT was estimated as {xbar:.4f}. The 95% confidence interval ({ci_mean_lower:.4f}, {ci_mean_upper:.4f}) provides a range of plausible values for the true population mean.

{"Because the benchmark value 65.0 lies within this confidence interval and the p-value is greater than 0.05, we do not have sufficient evidence to conclude that the population mean differs from 65.0." if mean_ci_contains_mu0 else "Because the benchmark value 65.0 does not lie within this confidence interval and the p-value is less than 0.05, we have sufficient evidence to conclude that the population mean differs from 65.0."}

This result is consistent with the EDA findings. Although the histogram and boxplot suggested that many BMIPCT values were relatively high and the median was above 65, the data also showed substantial variability, which reduces the strength of the apparent difference from the benchmark.

Therefore, the population mean BMIPCT appears to be consistent with 65.0, and the observed sample difference is likely due to natural variation rather than a meaningful population-level difference.

Final Synthesis
---------------
This project used one-sample inference to examine whether the population proportion of current cigarette use differs from 0.20 and whether the population mean BMIPCT differs from 65.0.

The EDA provided important context for both analyses. For CurrentCigaretteUse, the sample proportion was already close to 0.20, although smoking prevalence varied across age groups. For BMIPCT, the distribution showed moderate to high values with substantial spread, suggesting that variability would be important in the inferential results.

Overall, the inferential analyses showed that there is not sufficient statistical evidence to conclude that either the population proportion of current cigarette use differs from 0.20 or that the population mean BMIPCT differs from 65.0.

These results show the value of combining EDA with inference: EDA helps reveal patterns in the sample, while inference helps determine whether those patterns are strong enough to support conclusions about the population.
""".strip()

print(final_summary_text)

Project Cycle 2 Final Summary

Research Questions
------------------
1. Is the proportion of students who currently use cigarettes different from 0.20?
2. Is the mean BMIPCT different from 65.0?

Behavior Variable: CurrentCigaretteUse
--------------------------------------
Parameter: population proportion p
Benchmark: p0 = 0.20

Sample size: 13323
Sample proportion: 0.1943
95% confidence interval: (0.1876, 0.2010)
Test statistic (z): -1.6374
P-value: 0.101542
Decision at alpha = 0.05: Fail to reject H0

Interpretation:
The sample proportion of students who currently use cigarettes was estimated as 0.1943. The 95% confidence interval (0.1876, 0.2010) provides a range of plausible values for the true population proportion.

Because the benchmark value 0.20 lies within this confidence interval and the p-value is greater than 0.05, we do not have sufficient evidence to conclude that the population proportion differs from 0.20.

This result is consistent with the EDA findings, where the obs

In [23]:
# Save final summary text file
with open("../outputs/summary/final_summary.txt", "w", encoding="utf-8") as f:
    f.write(final_summary_text)

print("Saved: ../outputs/summary/final_summary.txt")

Saved: ../outputs/summary/final_summary.txt


In [24]:
print("All required inference outputs have been created:")
print("- ../outputs/tables/inference_summary_table.csv")
print("- ../outputs/summary/final_summary.txt")

All required inference outputs have been created:
- ../outputs/tables/inference_summary_table.csv
- ../outputs/summary/final_summary.txt
